# Adaptive inference-driven block mixed-precision Cholesky factorization

The notebook `jcf-gp-knee-xyz-global-budget-colab.ipynb` compares two numerical rules for choosing the storage precision of the off-diagonal blocks of the Cholesky factor in the adaptive block mixed-precision factorization of Salvaña (2026): the per-block error budget of the paper and a global error budget shared across the blocks. Both rules look only at the rounding error of a block. This notebook adds a third rule that also looks at the observed data. It asks whether a block whose rounding has little effect on the fit of the observed response can be stored in lower precision, and whether that buys likelihood accuracy per byte of factor storage beyond what the numerical rules give.

The test problem is the trivariate separable space-time Matérn emulator of the three components of the knee joint contact force over strides and gait points, whose covariance matrix has $N = 3 \times n \times 101$ rows for $n$ training strides. Four methods are compared on the same matrix at the same parameters: Method A, the per-block rule; Method B, the global numerical budget spent by the paced online rule and, where feasible, by the offline plan; Method C, the inference-driven rule, Algorithm 2 below, over a range of its statistical tolerance $\varepsilon$; and Method D, every block in fp64, the reference for likelihood accuracy. For each the notebook reports the precision mix, the stored size of the factor, the numerical and the statistical budget utilization, the factorization time with the time spent on precision selection, and the likelihood and prediction errors against fp64. All arithmetic is fp64 and the diagonal factor blocks are fp64; the formats are storage only.

The notebook has four parts. It loads the data, it defines the model and the four methods with the theory behind them, it fits the parameters and runs the comparison with a sweep over $\varepsilon$, and it reports the results in tables and five figures. It runs on any GPU runtime. In Colab, choose Runtime, then Change runtime type, then a GPU.

## Setup

In [ ]:
import gc, json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

assert torch.cuda.is_available(), "No GPU is attached. Choose Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{GPU_NAME}, {GPU_GB:.1f} GB")

## Data

The data are those of the first notebook. One row is one stride of the right leg, and there are 480 training strides from twelve participants and 140 test strides from three other participants. The inputs are three ground reaction force curves, five joint angle curves, the belt speed, the body mass and the height. The outputs used here are the three components of the right knee joint contact force computed by OpenSim, in body weights (BW), time normalized to 101 points of the gait cycle: $f_x$ is anterior-posterior (shear), $f_y$ is along the long axis of the tibia and negative in compression, and $f_z$ is medio-lateral.

The size of the comparison is set by `STRIDES`, the number of training strides, evenly spaced over the 480, and it is the only value that has to change to go from a small test to the full data. With 24 strides the matrix has $N = 7{,}272$ rows and the notebook runs in a few minutes on any GPU. With 480 strides it has $N = 145{,}440$ rows, the factor stored by the per-block rule is about 42 GB, and the comparison needs the 80 GB NVIDIA A100. The parameters are fitted on the same strides; for the full data, where a fit of the 145,440 matrix would take hours, `FIT_STRIDES` can be set to a smaller number to fit on a subset of the comparison strides. `EPSILONS` is the list of statistical tolerances swept for Method C, with `inf` the inactive constraint, and `EPSILON_MAIN` the one shown in the precision maps.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
STRIDES = 24                                               # training strides of the comparison; N = 3 x STRIDES x 101; 480 for the full data
FIT_STRIDES = None                                         # None: the parameters are fitted on the same strides as the comparison; a smaller number fits on a subset of them, for the full data
TEST_STRIDES = 20                                          # test strides used for the prediction check
EPSILONS = [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, np.inf]    # statistical tolerances of Method C; inf switches the statistical budget off
EPSILON_MAIN = 1e-5                                        # the tolerance shown in the precision maps and used by the optional fit under Method C
JOINT = "knee"                                             # "hip", "knee" or "ankle"
COMPONENTS = ["fx", "fy", "fz"]

x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_test = pd.read_csv("data/x-test.csv").iloc[:TEST_STRIDES].reset_index(drop=True)
y_test = pd.read_csv("data/y-test.csv").iloc[:TEST_STRIDES].reset_index(drop=True)
keep = np.unique(np.linspace(0, len(x_train) - 1, STRIDES).round().astype(int))
x_train, y_train = x_train.iloc[keep].reset_index(drop=True), y_train.iloc[keep].reset_index(drop=True)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

Each stride becomes the vector $\mathbf{x}_i$ of its eight input curves and three scalars, standardized on the training strides exactly as in the first notebook. The Gaussian process models the residual of each component curve, the curve minus the mean curve $m_j(t)$ of the training strides, divided by the standard deviation $s_j$ of the training curves of that component.

In [ ]:
GRID = np.arange(101)
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

def build_features(x, x_ref):
    cols = []
    for name in INPUT_CURVES:
        C, C_ref = curve(x, name), curve(x_ref, name)
        cols.extend(((C - C_ref.mean(axis=0)) / C_ref.std()).T)
    for name in ["belt_speed_mps", "mass_kg", "height_m"]:
        cols.append((x[name].to_numpy() - x_ref[name].mean()) / x_ref[name].std())
    return np.array(cols).T

F_train = build_features(x_train, x_train)
F_test = build_features(x_test, x_train)
Y_train = np.stack([curve(y_train, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])      # (3 components, strides, 101 gait points), in BW
Y_test = np.stack([curve(y_test, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])
mean_t = Y_train.mean(axis=1)                                                           # mean curve of each component
scale = Y_train.std(axis=(1, 2))                                                        # s_j, in BW
Z_train = (Y_train - mean_t[:, None, :]) / scale[:, None, None]                         # standardized residuals
fit_rows = np.arange(len(keep)) if FIT_STRIDES is None else np.unique(np.linspace(0, len(keep) - 1, min(FIT_STRIDES, len(keep))).round().astype(int))   # the fit strides
F_fit, Z_fit = F_train[fit_rows], Z_train[:, fit_rows, :]
Z_train.shape, Z_fit.shape

## Trivariate separable space-time Matérn model

Let $i = 1, \dots, n$ index the training strides, let $t \in \{0, 1, \dots, 100\}$ denote the percent of the gait cycle, and let $j \in \{1, 2, 3\}$ index the anterior-posterior, the axial and the medio-lateral component. The input of stride $i$ is the vector $\mathbf{x}_i \in \mathbb{R}^{811}$ and $y_{ij}(t)$ is component $j$ at gait point $t$ of stride $i$. Each component follows
$$
y_{ij}(t) = m_j(t) + s_j \big( f_j(\mathbf{x}_i, t) + \varepsilon_{ij}(t) \big), \qquad \varepsilon_{ij}(t) \sim \mathcal{N}(0, \sigma_{\varepsilon, j}^2),
$$
where $(f_1, f_2, f_3)$ is a trivariate Gaussian process. Write $h$ for the root mean squared difference between the standardized inputs of two strides and $u = |t - t'|$ for the lag between two gait points. The covariance function is
$$
\operatorname{Cov}\!\big(f_j(\mathbf{x}, t), f_k(\mathbf{x}', t')\big) = \beta_{jk}\, \sigma_j \sigma_k \;\, \gamma_{jk}\, \mathcal{M}\!\Big(\frac{h}{\ell_{jk}}\Big) \;\, \gamma_{t, jk}\, \mathcal{M}\!\Big(\frac{u}{\ell_{t, jk}}\Big), \qquad \mathcal{M}(d) = \Big(1 + \sqrt{5}\, d + \tfrac{5}{3} d^2\Big) e^{-\sqrt{5}\, d},
$$
with $\mathcal{M}$ the Matérn correlation function with smoothness $5/2$. Component $j$ has its own variance $\sigma_j^2$, input range $\ell_j$ and temporal range $\ell_{t, j}$, the parameters $\beta_{jk}$ form a correlation matrix, and for a pair of components
$$
\frac{1}{\ell_{jk}^2} = \frac{1}{2} \Big( \frac{1}{\ell_j^2} + \frac{1}{\ell_k^2} \Big), \qquad \gamma_{jk} = \Big( \frac{2\, \ell_j \ell_k}{\ell_j^2 + \ell_k^2} \Big)^{5/2},
$$
and likewise for the temporal ranges (Gneiting, Kleiber and Schlather, 2010; Apanasovich, Genton and Sun, 2012). The cross-correlation between components $j$ and $k$ at the same stride and gait point is $\rho_{jk} = \beta_{jk}\, \gamma_{jk}\, \gamma_{t, jk}$.

The model has fourteen parameters. Writing $\boldsymbol{\Sigma} = \sigma_1^2 \bar{\boldsymbol{\Sigma}}$, the variance of the first component is profiled out, $\hat{\sigma}_1^2 = \mathbf{y}^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y} / N$, and the negative log likelihood of the stacked residuals $\mathbf{y}$ is
$$
-\log p = \tfrac{1}{2} \Big( N \log \hat{\sigma}_1^2 + \log \det \bar{\boldsymbol{\Sigma}} + N + N \log 2\pi \Big).
$$
The matrix $\bar{\boldsymbol{\Sigma}}$ is a nonnegative definite matrix plus the diagonal matrix of the relative nugget effects $\eta_j = \sigma_{\varepsilon, j}^2 / \sigma_1^2$, so $m = \min_j \eta_j$ is a lower bound on its smallest eigenvalue, available at no cost. The class below orders the points by component, then stride, then gait point, and returns any block of $\bar{\boldsymbol{\Sigma}}$ on the GPU in double precision, so the full matrix is never stored.

In [ ]:
def matern(d):
    s = 5.0**0.5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)


def correlation_matrix(raw):
    # 3 x 3 correlation matrix from three unconstrained numbers, through the angles of its Cholesky factor
    a = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.array([[1.0, 0.0, 0.0],
                  [np.sin(a[0]), np.cos(a[0]), 0.0],
                  [np.sin(a[1]), np.cos(a[1]) * np.sin(a[2]), np.cos(a[1]) * np.cos(a[2])]])
    return L @ L.T


def pair_ranges(l):
    # 1 / l_jk and gamma_jk for every pair of components
    a2 = 1.0 / l**2
    inverse = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    return inverse, (np.sqrt(a2[:, None] * a2[None, :]) / inverse**2) ** 2.5


class SpaceTimeTrivariateMatern:
    def __init__(self, F, Z, t, block_size):
        self.n, self.T, self.N = Z.shape[1], Z.shape[2], Z.size
        Ft = torch.tensor(F, dtype=torch.float64, device=DEVICE)
        self.F = Ft
        self.H = torch.cdist(Ft, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F.shape[1]**0.5      # h between strides
        self.t = torch.tensor(t, dtype=torch.float64, device=DEVICE)
        self.y = torch.tensor(Z.reshape(-1), dtype=torch.float64, device=DEVICE)                          # stacked residuals
        index = torch.arange(self.N, device=DEVICE)
        self.component, self.stride, self.time = index // (self.n * self.T), (index // self.T) % self.n, index % self.T
        self.blocks = [slice(a, min(a + block_size, self.N)) for a in range(0, self.N, block_size)]

    def set_parameters(self, theta):
        # theta = log input ranges (3), log temporal ranges (3), correlation angles (3), log variance ratios of fy and fz (2), log relative nugget effects (3)
        l, lt = np.exp(theta[0:3]), np.exp(theta[3:6])
        beta = correlation_matrix(theta[6:9])
        v = np.concatenate([[1.0], np.exp(theta[9:11])])
        eta = np.exp(theta[11:14])
        inverse_l, gamma = pair_ranges(l)
        inverse_lt, gamma_t = pair_ranges(lt)
        rho = beta * gamma * gamma_t                                               # cross-correlations at the same stride and gait point
        as_tensor = lambda a: torch.tensor(a, dtype=torch.float64, device=DEVICE)
        self.amplitude = as_tensor(rho * np.sqrt(np.outer(v, v)))
        self.inverse_l, self.inverse_lt = as_tensor(inverse_l), as_tensor(inverse_lt)
        self.nugget = as_tensor(eta)[self.component]                               # relative nugget effect of every point
        self.m = float(eta.min())                                                  # lower bound on the smallest eigenvalue
        self.parameters = dict(l=l, lt=lt, v=v, eta=eta, rho=rho)

    def block(self, rows, cols):
        # block of the covariance matrix between two sets of points, without the nugget effect
        pair = (self.component[rows][:, None], self.component[cols][None, :])
        h = self.H[self.stride[rows]][:, self.stride[cols]]
        u = (self.t[self.time[rows]][:, None] - self.t[self.time[cols]][None, :]).abs()
        return self.amplitude[pair] * matern(h * self.inverse_l[pair]) * matern(u * self.inverse_lt[pair])

    def cross(self, F_new, rows_train):
        # covariance between every (component, new stride, gait point) and the training points in rows_train, without the nugget effect
        Fn = torch.tensor(F_new, dtype=torch.float64, device=DEVICE)
        h_new = torch.cdist(Fn, self.F, compute_mode="donot_use_mm_for_euclid_dist") / self.F.shape[1]**0.5    # (new strides, training strides)
        n_new = Fn.shape[0]
        idx = torch.arange(3 * n_new * self.T, device=DEVICE)
        c, s, t = idx // (n_new * self.T), (idx // self.T) % n_new, idx % self.T
        pair = (c[:, None], self.component[rows_train][None, :])
        h = h_new[s][:, self.stride[rows_train]]
        u = (self.t[t][:, None] - self.t[self.time[rows_train]][None, :]).abs()
        return self.amplitude[pair] * matern(h * self.inverse_l[pair]) * matern(u * self.inverse_lt[pair])

## The factorization, the storage perturbation and the four precision rules

### The storage perturbation

Partition $\bar{\boldsymbol{\Sigma}}$ into an $n_B \times n_B$ grid of blocks. The factorization proceeds by block columns. All arithmetic is in `fp64`, the diagonal factor blocks are kept in `fp64`,
$$
\widetilde{\mathbf{L}}_{kk} = \operatorname{chol}\Big( \bar{\boldsymbol{\Sigma}}_{kk} - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \widetilde{\mathbf{L}}_{kj}^\top \Big),
$$
and each off-diagonal block is computed in `fp64` as
$$
\mathbf{A}_{ik} = \Big( \bar{\boldsymbol{\Sigma}}_{ik} - \sum_{j < k} \widetilde{\mathbf{L}}_{ij} \widetilde{\mathbf{L}}_{kj}^\top \Big) \widetilde{\mathbf{L}}_{kk}^{-\top}
$$
and stored in some format $p$, $\widetilde{\mathbf{L}}_{ik} = \mathrm{fl}_p(\mathbf{A}_{ik})$, with conversion error $\mathbf{E}_{ik,p} = \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik}$. Under exact `fp64` arithmetic the perturbation $\boldsymbol{\Delta} = \widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top - \bar{\boldsymbol{\Sigma}}$ of the reconstructed covariance matrix has zero diagonal blocks and off-diagonal blocks $\boldsymbol{\Delta}_{ik} = \mathbf{E}_{ik} \widetilde{\mathbf{L}}_{kk}^\top$, so
$$
\|\boldsymbol{\Delta}\|_F^2 = 2 \sum_{k} \sum_{i > k} \big\| \mathbf{E}_{ik} \widetilde{\mathbf{L}}_{kk}^\top \big\|_F^2 ,
$$
and by Weyl's inequality $\widetilde{\boldsymbol{\Sigma}}$ is positive definite whenever $\|\boldsymbol{\Delta}\|_F \le \tau m$ for a lower bound $m$ on $\lambda_{\min}(\bar{\boldsymbol{\Sigma}})$ and $0 < \tau < 1$. Applied to the partially stored matrix at every block column, the same bound makes every diagonal Cholesky exist, so the factorization cannot break down. The storage formats are `bf16` with 2 bytes per entry, `fp32` with 4 and `fp64` with 8; `fp8` can be added to `CANDIDATES` below as in the global-budget notebook.

### Method A, the per-block budget of Salvaña (2026)

Block $(i, k)$ is stored in the lowest format $p$, in the order `bf16`, `fp32`, that satisfies
$$
\big\| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big\|_F \, \big\| \widetilde{\mathbf{L}}_{kk} \big\|_2 \le b = \frac{\tau m}{n_B},
$$
and in `fp64` if none does. Since $\|\boldsymbol{\Delta}_{ik}\|_F \le b$ for every block, $\|\boldsymbol{\Delta}\|_F \le n_B b = \tau m$. The rule is online and conservative: it bounds the actual perturbation by a product of norms and reserves the same share for every block.

### Method B, the global numerical budget

The global rule keeps the running total of the realized contributions
$$
d_{ik, p}^2 = 2\, \big\| \mathbf{E}_{ik,p} \widetilde{\mathbf{L}}_{kk}^\top \big\|_F^2
$$
and admits a format only if the total stays within $B_N = \tau^2 m^2$. Two ways of spending the budget are used here. **B-paced** divides the budget left by the number of blocks left and stores the lowest format whose contribution is within that allowance, with unused budget rolling over; it is online and cannot be bankrupted by early blocks. **B-optimal** records the costs of every format for every block during the run of Method A, moves blocks from `fp32` to `bf16` in order of increasing extra cost per byte saved until a fraction `PLAN_FRACTION` of the budget is spent, and then factors again storing the planned formats while still checking every realized contribution against the running total. With equal block sizes the plan is the minimum-storage assignment for the recorded costs; the margin leaves room for the realized costs of the second pass.

### Method C, the inference-driven rule (Algorithm 2)

The numerical rules see only the rounding error of a block. The observed response $\mathbf{y}$, the stacked standardized residuals, can say how much that rounding matters for the fit. The block forward substitution of $\mathbf{y}$ is carried along the factorization at no extra factorization: at block column $k$,
$$
\mathbf{z}_k = \widetilde{\mathbf{L}}_{kk}^{-1} \mathbf{r}_k, \qquad \mathbf{r}_k = \mathbf{y}_k - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \mathbf{z}_j ,
$$
and after block $(i, k)$ is stored, $\mathbf{r}_i \leftarrow \mathbf{r}_i - \widetilde{\mathbf{L}}_{ik} \mathbf{z}_k$. When the factorization ends, $\mathbf{z} = \widetilde{\mathbf{L}}^{-1} \mathbf{y}$ and $\|\mathbf{z}\|^2 = \mathbf{y}^\top \widetilde{\boldsymbol{\Sigma}}^{-1} \mathbf{y}$ is the quadratic form of the likelihood; the notebook checks it against a separate solve.

Rounding $\mathbf{A}_{ik}$ to format $p$ changes the residual update of row block $i$ from $\mathbf{A}_{ik} \mathbf{z}_k$ to $\mathbf{A}_{ik} \mathbf{z}_k + \mathbf{E}_{ik,p} \mathbf{z}_k$, so the perturbation of the residual is
$$
\mathbf{e}_{ik,p} = \mathbf{E}_{ik,p} \mathbf{z}_k .
$$
This is the effect of the rounding on the observed data as the factorization sees them at that point: $\mathbf{z}_k$ is the innovation of block $k$, the part of $\mathbf{y}_k$ not explained by the earlier blocks, and $\mathbf{E}_{ik,p} \mathbf{z}_k$ is how much of it the rounded block passes on, wrongly, to block $i$.

**Derivation of the score.** Conditionally on the blocks before it, row block $i$ contributes $\mathbf{r}_i^\top \mathbf{S}_i^{-1} \mathbf{r}_i$ to the quadratic form, with $\mathbf{S}_i$ its Schur complement and $\mathbf{r}_i$ its residual after the earlier blocks. Replace $\mathbf{S}_i$ by its diagonal $\mathbf{v}_i$, which is available during the factorization: it starts at $\operatorname{diag}(\bar{\boldsymbol{\Sigma}}_{ii})$, nugget included, and loses the row sums of squares of every stored block of that row, $\mathbf{v}_i \leftarrow \mathbf{v}_i - \operatorname{diag}(\widetilde{\mathbf{L}}_{ik} \widetilde{\mathbf{L}}_{ik}^\top)$. The approximation is $Q_i = \sum_a r_{ia}^2 / v_{ia}$. Let $\mathbf{r}_i^{(0)} = \mathbf{r}_i - \mathbf{A}_{ik} \mathbf{z}_k$ be the residual after an exact update. The stored block gives $\mathbf{r}_i^{(0)} - \mathbf{e}_{ik,p}$ instead, and the half quadratic form changes by
$$
\sum_a \frac{(r^{(0)}_{ia} - e_a)^2 - (r^{(0)}_{ia})^2}{2 v_{ia}} = \sum_a \frac{-\, r^{(0)}_{ia} e_a + e_a^2 / 2}{v_{ia}},
$$
whose absolute value is at most
$$
s_{ik,p} = \sum_{a=1}^{n_i} \frac{\big| r^{(0)}_{ia}\, e_{ik,p,a} \big| + \tfrac12 e_{ik,p,a}^2}{v_{ia}} \tag{1}
$$
by the triangle inequality. The score bounds the change that rounding one block causes in the diagonal conditional-residual approximation of the half quadratic form, in units of the standardized residuals, where the profiled variance is close to one, so $s$ reads in nats. It is a sensitivity proxy and not a bound on the likelihood: the later residual updates, the later diagonal factorizations and the interactions between rounded blocks are outside it, and the diagonal of the Schur complement stands in for the full conditional covariance. The score is evaluated with the $\mathbf{v}_i$ available before the candidate block is stored, and entries of $\mathbf{v}_i$ that are not positive are replaced by $m$, which can only increase the score. Formats whose conversion produces nonfinite entries are rejected.

**Two paced budgets.** Method C keeps the numerical budget $B_N = \tau^2 m^2$ on the contributions $d^2$ and adds a statistical budget $B_S = \varepsilon$ on the scores $s$. With $q$ off-diagonal blocks left, including the current one, the allowances are
$$
b_N = \frac{B_N - B_N^{\mathrm{used}}}{q}, \qquad b_S = \frac{B_S - B_S^{\mathrm{used}}}{q},
$$
and the lowest format with $d_{ik,p}^2 \le b_N$ and $s_{ik,p} \le b_S$ is stored, `fp64` if none qualifies; the two totals are then updated and unused budget rolls over. With $\varepsilon = \infty$ the statistical budget is inactive and the rule is B-paced, which is the ablation that isolates the effect of the data. The rule is an online heuristic, not an optimal allocation, and since it adds a restriction to the numerical budget it cannot store less than B-paced; what it offers is control of the data fit, and the experiment measures whether that buys likelihood accuracy per byte.

```text
Algorithm 2  Adaptive inference-driven block mixed-precision Cholesky
  B_N = (tau m)^2, B_S = epsilon, used_N = used_S = 0
  for every block i: r_i = y_i, v_i = diag(Sigma_ii)
  for k = 1, ..., n_B:
      L_kk = chol(Sigma_kk - sum_{j<k} L_kj L_kj^T), stored in fp64;  logdet += 2 sum log diag L_kk
      z_k = L_kk^{-1} r_k
      for i = k+1, ..., n_B:
          A_ik = (Sigma_ik - sum_{j<k} L_ij L_kj^T) L_kk^{-T}            in fp64
          r0 = r_i - A_ik z_k;  q = blocks left;  b_N = (B_N - used_N)/q;  b_S = (B_S - used_S)/q
          for p in bf16, fp32:  E = fl_p(A_ik) - A_ik;  e = E z_k
                                s = sum_a (|r0_a e_a| + e_a^2/2) / v_ia;  d2 = 2 ||E L_kk^T||_F^2
                                accept the first finite p with d2 <= b_N and s <= b_S
          if none accepted: p = fp64 (d2 = s = 0)
          L_ik = fl_p(A_ik);  used_N += d2;  used_S += s
          r_i -= L_ik z_k;  v_i -= rowsum(L_ik^2)
```

### Guarantee and what it does not cover

Every variant of Methods B and C accepts a block only if the running numerical total stays within $\tau^2 m^2$. Under exact `fp64` arithmetic and storage, the matrix of the blocks stored so far is therefore always within $\tau m$ of $\bar{\boldsymbol{\Sigma}}$ in Frobenius norm, the induction of Propositions 1 and 2 of the paper applies unchanged, every diagonal Cholesky exists, and the completed factor satisfies $\|\widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top - \bar{\boldsymbol{\Sigma}}\|_F \le \tau m < \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$, so $\widetilde{\boldsymbol{\Sigma}}$ is positive definite by Weyl's inequality. The statistical budget only removes candidates and cannot break this. The implementation runs in finite-precision `fp64`, whose rounding is not covered by the bound; the dense verification below checks the identity for $\|\boldsymbol{\Delta}\|_F$ and the smallest eigenvalue directly when $N$ is small enough.

In [ ]:
TAU = 0.99                                                 # error-budget parameter
PLAN_FRACTION = 0.9                                        # share of the numerical budget the offline plan of B-optimal may use
N = 3 * len(keep) * len(GRID)
BLOCK_SIZE = 101 * max(3, int(np.ceil(3 * len(keep) / 144)))   # whole strides of one component per block, at most 144 block rows
RUN_REFERENCE = N * (N + 1) / 2 * 8 < 0.5 * GPU_GB * 1e9   # Method D needs an fp64 factor in memory
RUN_OPTIMAL = True                                         # B-optimal costs one extra factorization
VERIFY_DENSE = N <= 12000                                  # L~ L~^T - Sigma is formed only when the dense matrices fit comfortably
fp32_factor_gb = (N * (N + 1) / 2 + N * BLOCK_SIZE / 2) * 4 / 1e9
print(f"N = {N:,}, block size {BLOCK_SIZE}, {int(np.ceil(N / BLOCK_SIZE))} block rows; factor stored in fp32 about {fp32_factor_gb:.1f} GB; "
      f"fp64 reference {'included' if RUN_REFERENCE else 'skipped'}; dense verification {'on' if VERIFY_DENSE else 'off'}")
assert fp32_factor_gb < 0.75 * GPU_GB, f"the factor needs about {fp32_factor_gb:.0f} GB and this GPU has {GPU_GB:.0f} GB; reduce STRIDES or use the 80 GB A100"

CANDIDATES = {"bf16": torch.bfloat16, "fp32": torch.float32}      # lowest precision first
LARGEST = {"bf16": 3.39e38, "fp32": 3.40e38}                       # largest finite number of each format
FORMAT_NAMES = ["bf16", "fp32", "fp64"]
BYTES = {"bf16": 2, "fp32": 4, "fp64": 8}


def block_costs(A, Lkk):
    """Conversion error of A in every candidate format: the error matrix E_p, its Frobenius norm (per-block rule), and the
    realized contribution d^2 = 2 ||E_p L_kk^T||_F^2 to the squared Frobenius norm of the covariance perturbation (global
    budget). Overflowing formats are infeasible. fp64 is the reference and costs nothing."""
    largest = float(A.abs().max())
    E, err, d2, finite = {}, {}, {}, {}
    for name, dtype in CANDIDATES.items():
        Ep = A.to(dtype).to(torch.float64) - A
        finite[name] = largest <= LARGEST[name] and bool(torch.isfinite(Ep).all())
        E[name] = Ep if finite[name] else None
        err[name] = float(torch.linalg.norm(Ep)) if finite[name] else np.inf
        d2[name] = 2.0 * float(torch.linalg.norm(Ep @ Lkk.T))**2 if finite[name] else np.inf
    E["fp64"], err["fp64"], d2["fp64"], finite["fp64"] = None, 0.0, 0.0, True
    return E, err, d2, finite


def sensitivity(E, z_k, r0, v_i, m):
    """Inference-sensitivity score s = sum_a (|r0_a e_a| + e_a^2 / 2) / v_a with e = E z_k, the perturbation of the residual
    update of row block i by the rounding of block (i, k). Entries of v_i that are not positive are replaced by m."""
    if E is None:
        return 0.0, 0.0
    e = E @ z_k
    v = torch.where(torch.isfinite(v_i) & (v_i > 0), v_i, torch.full_like(v_i, m))
    return float((((r0 * e).abs() + 0.5 * e * e) / v).sum()), float(e.norm())

The rules share one interface: for the current block they receive the conversion errors, the numerical contributions $d^2$ and the scores $s$ of every candidate format, and return the format to store. The offline plan of B-optimal is built by `plan_optimal` from the costs recorded during the run of Method A.

In [ ]:
class PerBlockRule:
    """Method A: the original rule, ||fl_p(A) - A||_F ||L_kk||_2 <= tau m / n_B."""
    name = "A: per-block budget"

    def __init__(self, m, nB, n_blocks, **kw):
        self.b = TAU * m / nB

    def choose(self, info):
        for name in CANDIDATES:
            if info["finite"][name] and info["err"][name] * info["norm_kk"] <= self.b:
                return name
        return "fp64"


class GlobalPaced:
    """Method B-paced: the lowest format whose numerical contribution is within the budget left divided by the blocks left."""
    name = "B-paced: global budget"

    def __init__(self, m, nB, n_blocks, **kw):
        self.budget, self.used, self.left = (TAU * m)**2, 0.0, n_blocks

    def choose(self, info):
        allowance = (self.budget - self.used) / self.left
        self.left -= 1
        for name in CANDIDATES:
            if info["finite"][name] and info["d2"][name] <= allowance:
                self.used += info["d2"][name]
                return name
        return "fp64"


class GlobalPlanned:
    """Method B-optimal, second pass: the planned format when the running numerical budget allows it, else the next one."""
    name = "B-optimal: global budget"

    def __init__(self, m, nB, n_blocks, plan=None, **kw):
        self.budget, self.used, self.plan, self.fallbacks = (TAU * m)**2, 0.0, plan, 0

    def choose(self, info):
        start = FORMAT_NAMES.index(self.plan[info["i"], info["k"]])
        for name in FORMAT_NAMES[start:]:
            if info["finite"][name] and self.used + info["d2"][name] <= self.budget:
                if name != self.plan[info["i"], info["k"]]:
                    self.fallbacks += 1
                self.used += info["d2"][name]
                return name
        return "fp64"


class InferenceDriven:
    """Method C, Algorithm 2: two paced budgets, the numerical budget tau^2 m^2 on d^2 and the statistical budget epsilon on
    the inference-sensitivity score s; the lowest format within both allowances is stored, fp64 if none is."""

    def __init__(self, m, nB, n_blocks, epsilon=np.inf, **kw):
        self.budget_N, self.used_N = (TAU * m)**2, 0.0
        self.budget_S, self.used_S = epsilon, 0.0
        self.left, self.epsilon = n_blocks, epsilon
        self.name = f"C: inference-driven, eps={epsilon:g}"

    def choose(self, info):
        b_N = (self.budget_N - self.used_N) / self.left
        b_S = (self.budget_S - self.used_S) / self.left if np.isfinite(self.budget_S) else np.inf
        self.left -= 1
        for name in CANDIDATES:
            if info["finite"][name] and info["d2"][name] <= b_N and info["s"][name] <= b_S:
                self.used_N += info["d2"][name]
                self.used_S += info["s"][name]
                return name
        return "fp64"


class AllFp64:
    name = "D: fp64 reference"

    def __init__(self, *a, **kw):
        pass

    def choose(self, info):
        return "fp64"


def plan_optimal(costs, sizes, budget):
    """Offline allocation from recorded costs: every block starts in fp32, then blocks move to bf16 in order of increasing extra
    cost per byte saved until the budget is spent; the minimum-storage assignment for the recorded costs when the block sizes
    are equal."""
    plan, used = {}, 0.0
    for key in costs:
        plan[key] = "fp32" if np.isfinite(costs[key]["fp32"]) else "fp64"
        used += costs[key][plan[key]]
    if used > budget:
        for key in sorted(costs, key=lambda key: -costs[key]["fp32"]):
            if used <= budget:
                break
            if plan[key] == "fp32":
                used -= costs[key]["fp32"]; plan[key] = "fp64"
    order = sorted((key for key in costs if plan[key] == "fp32" and np.isfinite(costs[key]["bf16"])),
                   key=lambda key: (costs[key]["bf16"] - costs[key]["fp32"]) / (sizes[key] * (BYTES["fp32"] - BYTES["bf16"])))
    for key in order:
        extra = costs[key]["bf16"] - costs[key]["fp32"]
        if used + extra <= budget:
            used += extra; plan[key] = "bf16"
    return plan, used

The factorization below is Algorithm 1 of the paper with the recurrences of Algorithm 2 carried alongside: the innovations $\mathbf{z}_k$, the residuals $\mathbf{r}_i$ and the Schur-diagonal proxies $\mathbf{v}_i$. For every off-diagonal block it measures the errors, the contributions and the scores of all candidate formats, hands them to the rule, stores the chosen format, and accumulates the realized contribution and score. The time spent measuring the candidates and choosing is reported as the selection overhead. The block forward and back substitution solves with the stored factor, and the likelihood, the predictions and the dense check reuse it.

In [ ]:
def mixed_precision_cholesky(model, rule, record=None):
    """Block Cholesky factorization with the storage format of every off-diagonal block chosen by `rule`.

    The covariance blocks are formed in fp64 and discarded, all arithmetic is fp64, the diagonal factor blocks are kept in fp64.
    Alongside the recurrence it carries the block forward substitution of the observed residuals, z_k = L_kk^{-1} r_k and
    r_i <- r_i - L~_ik z_k, and the diagonal v_i of the Schur complement of every row block, so that the inference-sensitivity
    score of every candidate format is available to the rule before the block is stored. For every block the conversion
    errors, the numerical contributions d^2 and the scores s of all candidate formats are measured and passed to the rule; the
    contributions of the chosen format are accumulated. The time spent on measuring the candidates and choosing is reported as
    the selection overhead. If `record` is a dict, the costs of every block in every format are stored in it.
    """
    blocks, nB = model.blocks, len(model.blocks)
    y = model.y
    L, logdet, used_N, used_S, select_time = {}, 0.0, 0.0, 0.0, 0.0
    count = {name: 0 for name in FORMAT_NAMES}
    precision = np.full((nB, nB), np.nan)
    r = [y[b].clone() for b in blocks]                                             # observed-data residuals, per block
    v = [None] * nB                                                                # Schur-complement diagonals, per block
    z = [None] * nB                                                                # innovations z_k
    assignments, failure = [], None
    n_off = nB * (nB - 1) // 2
    left = n_off
    t_start = time.time()
    for k in range(nB):
        S = model.block(blocks[k], blocks[k])
        S.diagonal().add_(model.nugget[blocks[k]])
        if v[k] is None:
            v[k] = S.diagonal().clone()
        row = [L[k, j].to(torch.float64) for j in range(k)]
        for Lkj in row:
            S.addmm_(Lkj, Lkj.T, alpha=-1.0)
        try:
            Lkk = torch.linalg.cholesky(S)
        except Exception as error:                                                 # not expected under the guarantee; reported if it happens
            failure = f"diagonal block {k}: {str(error).splitlines()[0]}"
            break
        L[k, k] = Lkk
        precision[k, k] = FORMAT_NAMES.index("fp64")
        logdet += 2.0 * float(torch.log(torch.diagonal(Lkk)).sum())
        norm_kk = float(torch.linalg.matrix_norm(Lkk, ord=2))
        z[k] = torch.linalg.solve_triangular(Lkk, r[k].reshape(-1, 1), upper=False).reshape(-1)
        for i in range(k + 1, nB):
            if v[i] is None:
                v[i] = model.block(blocks[i], blocks[i]).diagonal() + model.nugget[blocks[i]]
            A = model.block(blocks[i], blocks[k])
            for j in range(k):
                A.addmm_(L[i, j].to(torch.float64), row[j].T, alpha=-1.0)
            A = torch.linalg.solve_triangular(Lkk.T, A, upper=True, left=False)
            t0 = time.time()
            E, err, d2, finite = block_costs(A, Lkk)
            r0 = r[i] - A @ z[k]                                                   # the fp64 candidate residual
            s, e_norm = {}, {}
            for name in FORMAT_NAMES:
                s[name], e_norm[name] = sensitivity(E[name], z[k], r0, v[i], model.m)
            info = dict(i=i, k=k, err=err, d2=d2, s=s, finite=finite, norm_kk=norm_kk, left=left)
            if record is not None:
                record[i, k] = dict(d2)
            name = rule.choose(info)
            select_time += time.time() - t0
            left -= 1
            Lik = A.to(CANDIDATES.get(name, torch.float64))
            L[i, k] = Lik
            Lik64 = Lik.to(torch.float64)
            r[i] = r[i] - Lik64 @ z[k]                                             # residual update with the stored block
            v[i] = v[i] - (Lik64 * Lik64).sum(dim=1)                               # Schur-complement diagonal proxy
            used_N += d2[name]
            used_S += s[name]
            count[name] += 1
            precision[i, k] = FORMAT_NAMES.index(name)
            assignments.append({"i": i, "k": k, "format": name, "d2": d2[name], "s": s[name], "e_norm": e_norm[name],
                                "d2_bf16": d2["bf16"], "d2_fp32": d2["fp32"], "s_bf16": s["bf16"], "s_fp32": s["fp32"],
                                "per_block_test_bf16": err["bf16"] * norm_kk, "per_block_test_fp32": err["fp32"] * norm_kk})
    elapsed = time.time() - t_start
    quadratic = float(sum((zk * zk).sum() for zk in z if zk is not None)) if failure is None else np.nan   # y^T Sigma~^{-1} y from the carried forward substitution
    return dict(L=L, logdet=logdet, count=count, precision=precision, used_N=used_N, used_S=used_S, bytes=sum(b.nbytes for b in L.values()),
                time=elapsed, select_time=select_time, assignments=assignments, failure=failure, quadratic=quadratic)


def solve(L, blocks, rhs):
    # solve (L L^T) x = rhs by block forward and back substitution; rhs is a vector or a matrix with one right-hand side per column
    x, nB = rhs.clone().reshape(rhs.shape[0], -1), len(blocks)
    for k in range(nB):
        s = x[blocks[k]]
        for j in range(k):
            s -= L[k, j].to(torch.float64) @ x[blocks[j]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k], s, upper=False)
    for k in range(nB - 1, -1, -1):
        s = x[blocks[k]]
        for i in range(k + 1, nB):
            s -= L[i, k].to(torch.float64).T @ x[blocks[i]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k].T, s, upper=True)
    return x.reshape(rhs.shape)


def negative_log_likelihood(theta, model):
    # Gaussian log likelihood of the replicates, which share the covariance matrix; one factorization, one solve per replicate
    model.set_parameters(theta)
    L, logdet, count, precision = mixed_precision_cholesky(model)
    alpha = solve(L, model.blocks, model.y)                                        # (N, replicates)
    N, T = model.N, model.T
    sigma2 = float((model.y * alpha).sum()) / (N * T)                              # profiled variance of the first output
    stored_gb = sum(block.nbytes for block in L.values()) / 1e9
    return 0.5 * (N * T * np.log(sigma2) + T * logdet + N * T + N * T * np.log(2 * np.pi)), sigma2, count, stored_gb, precision

def negative_log_likelihood_from_factor(model, result):
    # the profiled Gaussian log likelihood of the stacked residuals, evaluated with a given factor
    alpha = solve(result["L"], model.blocks, model.y)
    N = model.N
    sigma2 = float(model.y @ alpha) / N
    return 0.5 * (N * np.log(sigma2) + result["logdet"] + N + N * np.log(2 * np.pi)), sigma2

def predict(model, L, F_new, sigma2, chunk=8):
    # predictive mean and standard deviation of the three standardized component curves of every new stride, in chunks of strides
    alpha = solve(L, model.blocks, model.y)
    all_rows = torch.arange(model.N, device=DEVICE)
    means, sds = [], []
    eta = torch.tensor(model.parameters["eta"], device=DEVICE)
    for a in range(0, F_new.shape[0], chunk):
        K_star = model.cross(F_new[a:a + chunk], all_rows)                                                 # (3 n_chunk 101, N)
        n_chunk = min(chunk, F_new.shape[0] - a)
        mean = (K_star @ alpha).reshape(3, n_chunk, model.T)
        V = solve(L, model.blocks, K_star.T)                                                                # (N, 3 n_chunk 101)
        explained = (K_star * V.T).sum(dim=1).reshape(3, n_chunk, model.T)
        prior = (torch.diagonal(model.amplitude) + eta)[:, None, None]
        var = sigma2 * (prior - explained)
        means.append(mean.cpu().numpy()); sds.append(np.sqrt(np.maximum(var.cpu().numpy(), 1e-12)))
    return np.concatenate(means, axis=1), np.concatenate(sds, axis=1)

In [ ]:
def dense_check(model, result):
    # forms L~ L~^T - Sigma, for verification only, and returns its Frobenius norm and the smallest eigenvalues
    blocks, nB = model.blocks, len(model.blocks)
    Sigma = torch.cat([torch.cat([model.block(blocks[i], blocks[j]) for j in range(nB)], dim=1) for i in range(nB)], dim=0)
    Sigma.diagonal().add_(model.nugget)
    Lt = torch.zeros_like(Sigma)
    for (i, k), block in result["L"].items():
        Lt[blocks[i], blocks[k]] = block.to(torch.float64)
    Delta = Lt @ Lt.T - Sigma
    return float(torch.linalg.norm(Delta)), float(torch.linalg.eigvalsh(Sigma)[0]), float(torch.linalg.eigvalsh(Lt @ Lt.T)[0])

### Size of the covariance matrix

In [ ]:
model = SpaceTimeTrivariateMatern(F_train, Z_train, GRID.astype(float), BLOCK_SIZE)
N, nB = model.N, len(model.blocks)
n_off = nB * (nB - 1) // 2
sizes = {(i, k): (model.blocks[i].stop - model.blocks[i].start) * (model.blocks[k].stop - model.blocks[k].start) for k in range(nB) for i in range(k + 1, nB)}
print(f"covariance matrix          {N:,} x {N:,}   (1 joint x {len(COMPONENTS)} components x {model.n} strides x {model.T} gait points)")
print(f"matrix in fp64             {N * N * 8 / 1e9:.2f} GB, Cholesky factor in fp64 {N * (N + 1) / 2 * 8 / 1e9:.2f} GB")
print(f"blocks                     {nB} x {nB} of up to {BLOCK_SIZE} points, {n_off} off-diagonal factor blocks")
print(f"GPU                        {GPU_NAME}, {GPU_GB:.1f} GB")

## Maximum likelihood estimation

The fourteen parameters are estimated by L-BFGS-B with forward-difference gradients, as in the earlier notebooks, with the per-block rule of Method A inside every evaluation of the likelihood, on the comparison strides or on the `FIT_STRIDES` subset of them. The estimates are saved to a checkpoint and reused. The comparison then runs every method at these fixed parameters, so that the differences in precision mix, storage and accuracy come from the rule alone. With `FIT_EACH_RULE = True` the parameters are also estimated under B-paced and under Method C at `EPSILON_MAIN`, each with its own factorization inside the likelihood, and the estimates are reported side by side.

In [ ]:
MAX_ITERATIONS = 20
FIT_EACH_RULE = False                                      # also estimate the parameters under B-paced and Method C at EPSILON_MAIN
fit_model = model if FIT_STRIDES is None else SpaceTimeTrivariateMatern(F_fit, Z_fit, GRID.astype(float), 101 * max(3, int(np.ceil(3 * len(fit_rows) / 144))))
fit_nB = len(fit_model.blocks)
fit_n_off = fit_nB * (fit_nB - 1) // 2
fit_sizes = {(i, k): (fit_model.blocks[i].stop - fit_model.blocks[i].start) * (fit_model.blocks[k].stop - fit_model.blocks[k].start) for k in range(fit_nB) for i in range(k + 1, fit_nB)}

# starting values (fx, fy, fz): the univariate knee input and temporal ranges, unit variance ratios, 0.1 % nugget effects
l0 = np.full(3, {"hip": 0.354, "knee": 0.287, "ankle": 0.313}[JOINT])
lt0 = np.full(3, {"hip": 5.57, "knee": 4.34, "ankle": 4.46}[JOINT])
theta0 = np.concatenate([np.log(l0), np.log(lt0), [0.3, 0.3, 0.3], [0.0, 0.0], np.log([1e-3, 1e-3, 1e-3])])
bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(np.log(1.0), np.log(30.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.1))] * 3


def make_rule(name, mdl, epsilon=None, plan=None):
    nb_, n_off_ = len(mdl.blocks), len(mdl.blocks) * (len(mdl.blocks) - 1) // 2
    if name == "A: per-block budget":
        return PerBlockRule(mdl.m, nb_, n_off_)
    if name == "B-paced: global budget":
        return GlobalPaced(mdl.m, nb_, n_off_)
    if name == "B-optimal: global budget":
        return GlobalPlanned(mdl.m, nb_, n_off_, plan=plan)
    if name == "D: fp64 reference":
        return AllFp64()
    return InferenceDriven(mdl.m, nb_, n_off_, epsilon=epsilon)


def fit_parameters(rule_name, epsilon, verbose):
    # maximum likelihood estimation with the named rule inside every likelihood evaluation, on fit_model
    tag = rule_name.split(":")[0].replace(" ", "") + (f"_eps{epsilon:g}" if epsilon is not None else "")
    checkpoint = f"spacetime_checkpoint_{JOINT}_xyz_{fit_model.n}strides_{tag}.json"
    if os.path.exists(checkpoint):
        saved = json.load(open(checkpoint))
        print(f"{rule_name}: parameters from {checkpoint}")
        return np.array(saved["theta"]), saved["nll"], saved.get("evaluations", np.nan), saved.get("time_s", np.nan)
    best = {"nll": np.inf, "theta": None, "evaluations": 0}
    t_fit = time.time()

    def objective(th):
        if best["theta"] is not None and np.max(np.abs(th - best["theta"])) > 0.7:
            return best["nll"] + 1e6 * float(np.max(np.abs(th - best["theta"])))
        t0 = time.time()
        fit_model.set_parameters(th)
        r = mixed_precision_cholesky(fit_model, make_rule(rule_name, fit_model, epsilon))
        if r["failure"] is not None:
            return 1e30
        nll, sigma2 = negative_log_likelihood_from_factor(fit_model, r)
        best["evaluations"] += 1
        if nll < best["nll"]:
            best.update(nll=nll, theta=th.copy())
            json.dump({"theta": th.tolist(), "nll": nll, "evaluations": best["evaluations"], "time_s": time.time() - t_fit}, open(checkpoint, "w"))
        if verbose or best["evaluations"] % 50 == 0:
            p = fit_model.parameters
            print(f"{rule_name:34s} evaluation {best['evaluations']:4d}  negative log likelihood {nll:13.2f}  blocks {'/'.join(FORMAT_NAMES)} {'/'.join(str(c) for c in r['count'].values())}  [{time.time() - t0:.1f} s]  "
                  f"input ranges {np.round(p['l'], 3).tolist()}  temporal ranges {np.round(p['lt'], 2).tolist()}  cross-correlations fx-fy {p['rho'][0, 1]:.3f}, fx-fz {p['rho'][0, 2]:.3f}, fy-fz {p['rho'][1, 2]:.3f}", flush=True)
        return nll

    result = minimize(objective, theta0, method="L-BFGS-B", bounds=bounds, options={"maxiter": MAX_ITERATIONS, "eps": 1e-3})
    elapsed = time.time() - t_fit
    json.dump({"theta": best["theta"].tolist(), "nll": best["nll"], "evaluations": best["evaluations"], "time_s": elapsed}, open(checkpoint, "w"))
    print(f"{rule_name}: {result.message}, {result.nit} iterations, {best['evaluations']} likelihood evaluations, {elapsed / 60:.1f} min; checkpoint {checkpoint}", flush=True)
    return best["theta"], best["nll"], best["evaluations"], elapsed


theta, nll_A, _, _ = fit_parameters("A: per-block budget", None, verbose=True)
fits = {"A: per-block budget": theta}
if FIT_EACH_RULE:
    fit_rows_table = []
    for rule_name, eps in [("A: per-block budget", None), ("B-paced: global budget", None), (f"C: inference-driven, eps={EPSILON_MAIN:g}", EPSILON_MAIN)]:
        th, nll, evaluations, elapsed = fit_parameters(rule_name, eps, verbose=False)
        fit_model.set_parameters(th)
        r = mixed_precision_cholesky(fit_model, make_rule(rule_name, fit_model, eps))
        _, sigma2 = negative_log_likelihood_from_factor(fit_model, r)
        pp = fit_model.parameters
        fit_rows_table.append({"method": rule_name, "negative log likelihood": nll, "evaluations": evaluations, "fit time [min]": elapsed / 60,
                               **{f"input range {c}": pp["l"][j] for j, c in enumerate(COMPONENTS)}, **{f"temporal range {c}": pp["lt"][j] for j, c in enumerate(COMPONENTS)},
                               **{f"variance {c}": sigma2 * pp["v"][j] for j, c in enumerate(COMPONENTS)}, **{f"nugget effect {c}": sigma2 * pp["eta"][j] for j, c in enumerate(COMPONENTS)},
                               "cross-correlation fx-fy": pp["rho"][0, 1], "cross-correlation fx-fz": pp["rho"][0, 2], "cross-correlation fy-fz": pp["rho"][1, 2],
                               "blocks bf16 / fp32 / fp64 at the estimates": " / ".join(str(r["count"][f]) for f in FORMAT_NAMES)})
        del r
    estimates_by_method = pd.DataFrame(fit_rows_table).set_index("method").T
    estimates_by_method.to_csv(f"inference_driven_estimates_by_method_{JOINT}_xyz_{fit_model.n}strides.csv")
    display(estimates_by_method)
if fit_model is not model:
    del fit_model
gc.collect(); torch.cuda.empty_cache()
model.set_parameters(theta)
p = model.parameters
budget_N = (TAU * model.m)**2
print(f"\ncomparison parameters (Method A): input ranges {np.round(p['l'], 3).tolist()}   temporal ranges {np.round(p['lt'], 2).tolist()}   relative nugget effects {[f'{x:.2e}' for x in p['eta']]}   m = {model.m:.3e}")
print(f"cross-correlations         fx-fy {p['rho'][0, 1]:.3f}, fx-fz {p['rho'][0, 2]:.3f}, fy-fz {p['rho'][1, 2]:.3f}")
print(f"per-block budget           b = tau m / n_B = {TAU * model.m / nB:.3e};   numerical budget tau^2 m^2 = {budget_N:.3e} (tau m = {TAU * model.m:.3e});   statistical tolerances {EPSILONS}")

## Comparison at fixed parameters

Every method factors the same matrix at the same parameters, one after another with one factor on the GPU at a time: Method D if its fp64 factor fits, Method A, B-paced, B-optimal, and Method C at every tolerance in `EPSILONS`. For each the cell records the precision mix, the stored size of the factor in bytes with the real dimensions and data types of every stored block, the realized numerical perturbation against its bound and the share of $\tau^2 m^2$ used, the accumulated score and the share of $\varepsilon$ used, the factorization time and the selection overhead inside it, the negative log likelihood, the largest change in the predicted test curves, and, when $N$ is small, the dense verification.

In [ ]:
def size(n_bytes):
    return f"{n_bytes / 1e9:.2f} GB" if n_bytes >= 1e9 else f"{n_bytes / 1e6:.2f} MB"

all_block_entries = sum(sizes.values()) + sum((model.blocks[k].stop - model.blocks[k].start)**2 for k in range(nB))
fp64_bytes = all_block_entries * 8                                                  # the same blocks, all stored in fp64 (diagonal blocks as full squares)
results, rows, all_assignments, reference = {}, [], [], {}

def evaluate(name, rule, epsilon=np.nan, record=None):
    r = mixed_precision_cholesky(model, rule, record)
    row = {"method": name, "epsilon": epsilon, "completed": r["failure"] is None, "failure": r["failure"] or "",
           **{f"blocks_{f}": r["count"][f] for f in FORMAT_NAMES}, **{f"share_{f}_%": 100 * r["count"][f] / n_off for f in FORMAT_NAMES},
           "stored_factor_bytes": r["bytes"], "stored_factor_MB": r["bytes"] / 1e6, "reduction_vs_fp64_%": 100 * (1 - r["bytes"] / fp64_bytes),
           "numerical_error_sqrt_sum_d2": np.sqrt(r["used_N"]), "numerical_error_over_tau_m": np.sqrt(r["used_N"]) / (TAU * model.m),
           "numerical_budget_used_%": 100 * r["used_N"] / budget_N, "within_numerical_budget": r["used_N"] <= budget_N,
           "statistical_sensitivity_sum_s": r["used_S"], "statistical_budget_used_%": 100 * r["used_S"] / epsilon if np.isfinite(epsilon) and epsilon > 0 else np.nan,
           "factorization_time_s": r["time"], "selection_overhead_s": r["select_time"], "quadratic_form_carried": r["quadratic"]}
    if isinstance(rule, GlobalPlanned):
        row["fallbacks_in_pass2"] = rule.fallbacks
    if r["failure"] is None:
        nll, sigma2 = negative_log_likelihood_from_factor(model, r)
        row["negative_log_likelihood"], row["profiled_sigma2"], row["quadratic_form_solve"] = nll, sigma2, sigma2 * N
        mean, sd = predict(model, r["L"], F_test, sigma2)
        if "pred_mean" not in reference:
            reference["pred_mean"] = mean
        row["max_prediction_difference_vs_first_BW"] = float(np.max(np.abs(scale[:, None, None] * (mean - reference["pred_mean"]))))
        if VERIFY_DENSE:
            dn, lam_min, lam_min_tilde = dense_check(model, r)
            row.update(dense_frobenius_norm=dn, dense_over_tracked=dn / np.sqrt(r["used_N"]) if r["used_N"] > 0 else np.nan, lambda_min_Sigma=lam_min, lambda_min_Sigma_tilde=lam_min_tilde)
    results[name] = {k: r[k] for k in ("count", "precision", "used_N", "used_S", "bytes", "time", "failure")}
    all_assignments.append(pd.DataFrame(r["assignments"]).assign(method=name, epsilon=epsilon))
    del r
    gc.collect(); torch.cuda.empty_cache()
    print(f"{name:34s} blocks {'/'.join(str(row[f'blocks_{f}']) for f in FORMAT_NAMES)}  stored {size(row['stored_factor_bytes'])}  numerical {row['numerical_error_over_tau_m']:.4f} tau m  "
          f"sum s {row['statistical_sensitivity_sum_s']:.3e}  [{row['factorization_time_s']:.1f} s, of which selection {row['selection_overhead_s']:.1f} s]", flush=True)
    return row

costs = {}
if RUN_REFERENCE:
    rows.append(evaluate("D: fp64 reference", AllFp64()))
rows.append(evaluate("A: per-block budget", PerBlockRule(model.m, nB, n_off), record=costs))
rows.append(evaluate("B-paced: global budget", GlobalPaced(model.m, nB, n_off)))
if RUN_OPTIMAL:
    plan, planned_used = plan_optimal(costs, sizes, PLAN_FRACTION * budget_N)
    rule = GlobalPlanned(model.m, nB, n_off, plan=plan)
    row = evaluate("B-optimal: global budget", rule)
    row.update(planned_budget_used_pct=100 * planned_used / budget_N, planned_bf16=sum(1 for f in plan.values() if f == "bf16"))
    rows.append(row)
for eps in EPSILONS:
    rule = InferenceDriven(model.m, nB, n_off, epsilon=eps)
    rows.append(evaluate(rule.name, rule, epsilon=eps))

table = pd.DataFrame(rows).set_index("method")
if "D: fp64 reference" in table.index:
    table["nll_minus_fp64"] = table["negative_log_likelihood"] - table.loc["D: fp64 reference", "negative_log_likelihood"]
    table["abs_nll_error_vs_fp64"] = table["nll_minus_fp64"].abs()
table["reduction_vs_A_%"] = 100 * (1 - table["stored_factor_bytes"] / table.loc["A: per-block budget", "stored_factor_bytes"])
table.to_csv(f"inference_driven_comparison_{JOINT}_xyz_{model.n}strides.csv")
pd.concat(all_assignments).to_csv(f"inference_driven_blocks_{JOINT}_xyz_{model.n}strides.csv", index=False)

### Results

Precision mix, total factor storage, reductions relative to the same blocks in `fp64` and relative to Method A, the negative log likelihood and its difference from the fp64 reference, the numerical and the statistical budget utilization, and the factorization time with the selection overhead inside it, for Method A, the two global variants, Method C at `EPSILON_MAIN` and the fp64 reference.

In [ ]:
main_C = f"C: inference-driven, eps={EPSILON_MAIN:g}"
shown_cols = [n for n in ["A: per-block budget", "B-paced: global budget", "B-optimal: global budget", main_C, "D: fp64 reference"] if n in table.index]
def fmt(name, key, f="{:.3f}"):
    v = table.loc[name, key] if key in table.columns else np.nan
    return "" if pd.isna(v) else f.format(v)
report = pd.DataFrame({name: {
    "bf16 blocks [%]": fmt(name, "share_bf16_%", "{:.1f}"), "fp32 blocks [%]": fmt(name, "share_fp32_%", "{:.1f}"), "fp64 blocks [%]": fmt(name, "share_fp64_%", "{:.1f}"),
    "total factor storage": size(table.loc[name, "stored_factor_bytes"]),
    "storage reduction vs fp64 [%]": fmt(name, "reduction_vs_fp64_%", "{:.1f}"), "storage reduction vs original [%]": fmt(name, "reduction_vs_A_%", "{:.1f}"),
    "negative log likelihood": fmt(name, "negative_log_likelihood", "{:,.3f}"), "likelihood difference vs fp64": fmt(name, "nll_minus_fp64", "{:+.4f}"),
    "numerical budget utilization [%]": fmt(name, "numerical_budget_used_%", "{:.2f}"), "numerical error / (tau m)": fmt(name, "numerical_error_over_tau_m", "{:.4f}"),
    "statistical sensitivity sum s": fmt(name, "statistical_sensitivity_sum_s", "{:.3e}"), "statistical budget utilization [%]": fmt(name, "statistical_budget_used_%", "{:.2f}"),
    "factorization time [s]": fmt(name, "factorization_time_s", "{:.2f}"), "of which selection overhead [s]": fmt(name, "selection_overhead_s", "{:.2f}"),
    "completed": "yes" if table.loc[name, "completed"] else "no"} for name in shown_cols})
report.to_csv(f"inference_driven_report_{JOINT}_xyz_{model.n}strides.csv")
print(f"fp64 factor {size(fp64_bytes)} as stored ({nB} full diagonal blocks and {n_off} off-diagonal blocks); N = {N:,}; fixed parameters, same blocks, tau = {TAU}")
report

### Sensitivity to the statistical tolerance

Method C over the tolerances in `EPSILONS`: the precision mix, the storage, the shares of the two budgets used and the likelihood error against fp64. The row with $\varepsilon = \infty$ is the inactive statistical constraint and should reproduce B-paced block for block. Where the tolerance is tight, blocks that fail the statistical test in both low formats fall to `fp64`, so the storage can exceed that of Method A; where it is loose, the rule converges to B-paced.

In [ ]:
sweep = table[table.index.str.startswith("C:")].copy()
sweep_cols = ["epsilon", "blocks_bf16", "blocks_fp32", "blocks_fp64", "share_bf16_%", "stored_factor_MB", "reduction_vs_A_%", "numerical_budget_used_%", "statistical_sensitivity_sum_s", "statistical_budget_used_%"] + (["nll_minus_fp64"] if "nll_minus_fp64" in table.columns else []) + ["factorization_time_s"]
sweep[sweep_cols].to_csv(f"inference_driven_epsilon_sweep_{JOINT}_xyz_{model.n}strides.csv")
display(sweep[sweep_cols].style.format(precision=4))
a = all_assignments[1 if RUN_REFERENCE else 0]
print(f"scores s of the off-diagonal blocks in the run of Method A: bf16 median {a['s_bf16'].median():.3e}, max {a['s_bf16'].max():.3e}, sum {a['s_bf16'].sum():.3e};  fp32 median {a['s_fp32'].median():.3e}, sum {a['s_fp32'].sum():.3e}")
print(f"carried quadratic form y^T Sigma~^-1 y against the solve with the stored factor, largest relative difference over the methods: "
      f"{float(((table['quadratic_form_carried'] - table['quadratic_form_solve']).abs() / table['quadratic_form_solve']).max()):.1e}")

### Dense verification

For checking only, and only when $N$ is small enough to hold the matrix, which `VERIFY_DENSE` decides, the cell forms $\widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}}$ from the stored factor and compares its Frobenius norm with the tracked $\sqrt{\sum d^2}$, which should agree to rounding, and reports the smallest eigenvalue of $\widetilde{\boldsymbol{\Sigma}}$ against that of $\bar{\boldsymbol{\Sigma}}$, which should stay above $(1 - \tau)\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$.

In [ ]:
if VERIFY_DENSE:
    display(table[["dense_frobenius_norm", "numerical_error_sqrt_sum_d2", "dense_over_tracked", "lambda_min_Sigma", "lambda_min_Sigma_tilde"]].style.format("{:.4e}"))
print("The positive-definiteness guarantee assumes exact fp64 arithmetic and storage; the implementation uses finite-precision fp64. The score s is a sensitivity proxy, not a bound on the likelihood.")

### Figures

Figure 1, the storage format of every block of the factor under Method A, B-paced, B-optimal and Method C at `EPSILON_MAIN`, with identical colors. Figure 2, the precision mix of every run, with the stored size above each bar. Figure 3, the total factor storage. Figure 4, the absolute likelihood error against fp64 as a function of the stored size, with Method C over all tolerances as squares. Figure 5, the precision mix, the storage and the likelihood error of Method C as the tolerance varies, against the storage of the numerical rules.

In [ ]:
PRECISION_COLORS = {"bf16": "#1FBEC0", "fp32": "#F26F22", "fp64": "#4D4D4D"}
METHOD_COLORS = {"A": "#4D4D4D", "B-paced": "#0072B2", "B-optimal": "#56B4E9", "C": "#D55E00", "D": "#999999"}
cmap = ListedColormap([PRECISION_COLORS[f] for f in FORMAT_NAMES]); cmap.set_bad("white")
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

# figure 1: blockwise precision maps
map_names = [n for n in ["A: per-block budget", "B-paced: global budget", "B-optimal: global budget", main_C] if n in results]
fig, axes = plt.subplots(1, len(map_names), figsize=(4.6 * len(map_names), 5.6), constrained_layout=True)
for ax, name in zip(np.atleast_1d(axes), map_names):
    rr = results[name]
    ax.imshow(np.ma.masked_invalid(rr["precision"]), cmap=cmap, norm=norm, origin="upper", interpolation="nearest")
    ax.set_title(f"{name}\n" + "   ".join(f"{f} {100 * rr['count'][f] / n_off:.0f}%" for f in FORMAT_NAMES) + f"\n{size(rr['bytes'])}, numerical error {np.sqrt(rr['used_N']) / (TAU * model.m):.2f} tau m", fontsize=10)
    ax.set_xlabel("block column $k$")
np.atleast_1d(axes)[0].set_ylabel("block row $i$")
fig.legend(handles=[Patch(facecolor=PRECISION_COLORS[f], label=f) for f in FORMAT_NAMES], loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f"Storage precision of the off-diagonal factor blocks: per-block, global and inference-driven rules\n($N$ = {N:,} = 3 components $\\times$ {model.n} strides $\\times$ {model.T} gait points, $n_b$ = {BLOCK_SIZE}, {nB} $\\times$ {nB} blocks, $\\tau$ = {TAU}, diagonal blocks fp64)", fontsize=12, y=1.1)
fig.savefig(f"fig1_precision_maps_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight"); plt.show()

# figure 2: precision mix of every method
names = list(results)
labels = [n.replace(": ", "\n").replace("inference-driven, ", "") for n in names]
fig, ax = plt.subplots(figsize=(1.3 * len(names) + 3, 4.6), constrained_layout=True)
bottom = np.zeros(len(names))
for f in FORMAT_NAMES:
    share = np.array([100 * results[n]["count"][f] / n_off for n in names])
    ax.bar(range(len(names)), share, bottom=bottom, color=PRECISION_COLORS[f], label=f, width=0.65)
    for x, (sh, b0) in enumerate(zip(share, bottom)):
        if sh >= 5:
            ax.text(x, b0 + sh / 2, f"{sh:.0f}%", ha="center", va="center", fontsize=8, color="white")
    bottom += share
for x, n in enumerate(names):
    ax.text(x, 102, size(results[n]["bytes"]), ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(names))); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("off-diagonal factor blocks [%]"); ax.set_ylim(0, 114); ax.set_yticks(range(0, 101, 20))
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.set_title("Precision mix of the stored Cholesky factor, with the stored size above each bar", loc="left", pad=10)
fig.savefig(f"fig2_precision_mix_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight"); plt.show()

# figure 3: total factor storage
fig, ax = plt.subplots(figsize=(1.3 * len(names) + 3, 4.2), constrained_layout=True)
vals = np.array([results[n]["bytes"] for n in names]) / 1e6
cols = [METHOD_COLORS["D" if n.startswith("D") else "A" if n.startswith("A") else "B-paced" if n.startswith("B-paced") else "B-optimal" if n.startswith("B-opt") else "C"] for n in names]
ax.bar(range(len(names)), vals, color=cols, width=0.65)
for x, vv in enumerate(vals):
    ax.text(x, vv, f"{vv:.1f}", ha="center", va="bottom", fontsize=8)
ax.axhline(fp64_bytes / 1e6, color="black", lw=0.8, ls=":", label="all blocks in fp64")
ax.set_xticks(range(len(names))); ax.set_xticklabels(labels, fontsize=8); ax.set_ylabel("stored Cholesky factor [MB]")
ax.legend(frameon=False, fontsize=8); ax.set_title("Total Cholesky factor storage, diagonal blocks included", loc="left")
fig.savefig(f"fig3_storage_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight"); plt.show()

# figure 4: likelihood accuracy against storage
if "abs_nll_error_vs_fp64" in table.columns:
    fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
    for n in names:
        if n.startswith("D"):
            continue
        key = "A" if n.startswith("A") else "B-paced" if n.startswith("B-paced") else "B-optimal" if n.startswith("B-opt") else "C"
        x, yv = table.loc[n, "stored_factor_MB"], max(table.loc[n, "abs_nll_error_vs_fp64"], 1e-6)
        ax.scatter(x, yv, color=METHOD_COLORS[key], s=55 if key != "C" else 40, marker="o" if key != "C" else "s", zorder=3)
        ax.annotate(n.replace("inference-driven, ", "").replace(": ", "\n") if key != "C" else f"eps={table.loc[n, 'epsilon']:g}", (x, yv), textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_yscale("log"); ax.set_xlabel("stored Cholesky factor [MB]"); ax.set_ylabel("|negative log likelihood - fp64| [nats]")
    ax.grid(alpha=0.3, which="both"); ax.set_title("Likelihood approximation error against factor storage (squares: Method C over epsilon)", loc="left", fontsize=10)
    fig.savefig(f"fig4_likelihood_vs_storage_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight"); plt.show()

# figure 5: sensitivity to epsilon
finite_eps = [e for e in EPSILONS if np.isfinite(e)]
xs = list(range(len(EPSILONS)))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
bottom = np.zeros(len(EPSILONS))
for f in FORMAT_NAMES:
    share = np.array([100 * results[f"C: inference-driven, eps={e:g}"]["count"][f] / n_off for e in EPSILONS])
    axes[0].bar(xs, share, bottom=bottom, color=PRECISION_COLORS[f], label=f, width=0.65); bottom += share
axes[0].set_xticks(xs); axes[0].set_xticklabels([f"{e:g}" if np.isfinite(e) else "inf" for e in EPSILONS]); axes[0].set_xlabel("statistical tolerance $\\varepsilon$")
axes[0].set_ylabel("off-diagonal factor blocks [%]"); axes[0].set_ylim(0, 100); axes[0].legend(frameon=False, ncol=3, fontsize=8, loc="lower left"); axes[0].set_title("Precision mix of Method C", loc="left")
st = [results[f"C: inference-driven, eps={e:g}"]["bytes"] / 1e6 for e in EPSILONS]
axes[1].plot(xs, st, "s-", color=METHOD_COLORS["C"], label="Method C")
for key, n in [("A", "A: per-block budget"), ("B-paced", "B-paced: global budget"), ("B-optimal", "B-optimal: global budget")]:
    if n in results:
        axes[1].axhline(results[n]["bytes"] / 1e6, color=METHOD_COLORS[key], ls="--", lw=1.2, label=n)
axes[1].set_xticks(xs); axes[1].set_xticklabels([f"{e:g}" if np.isfinite(e) else "inf" for e in EPSILONS]); axes[1].set_xlabel("statistical tolerance $\\varepsilon$")
axes[1].set_ylabel("stored Cholesky factor [MB]"); axes[1].grid(alpha=0.3); axes[1].legend(frameon=False, fontsize=8); axes[1].set_title("Storage of Method C against the numerical-only rules", loc="left")
if "abs_nll_error_vs_fp64" in table.columns:
    ax2 = axes[1].twinx()
    err = [max(table.loc[f"C: inference-driven, eps={e:g}", "abs_nll_error_vs_fp64"], 1e-6) for e in EPSILONS]
    ax2.plot(xs, err, "o:", color="#009E73", label="|nll - fp64| of Method C"); ax2.set_yscale("log"); ax2.set_ylabel("|negative log likelihood - fp64| [nats]", color="#009E73")
    ax2.legend(frameon=False, fontsize=8, loc="upper center")
fig.savefig(f"fig5_epsilon_sensitivity_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight"); plt.show()

### Reading the results

The fp64 reference sets the likelihood, and every mixed-precision run is measured against it. Method A spends a small fraction of a percent of the numerical budget, because its per-block share is tiny and `fp32` blocks contribute almost nothing; the global rules spend the same budget where it buys storage, B-paced online and B-optimal by plan. Method C adds the statistical budget on top of the numerical one. The sweep shows what it does: as $\varepsilon$ tightens, the likelihood error falls monotonically toward the fp64 value and the storage rises, first by refusing `bf16` and then by pushing blocks to `fp64`; as it loosens, the rule converges to B-paced. The question the experiment answers is whether, at a given storage, Method C reaches a smaller likelihood error than the numerical rules. That happens only if the score $s$ ranks the blocks differently from the numerical contribution $d^2$. Both are driven by the same error matrix $\mathbf{E}_{ik}$, so on a matrix where they rank the blocks alike, the Method C points lie on the line through Method A and B-paced in Figure 4 and the statistical budget acts as an accuracy dial rather than a source of extra memory savings; on a matrix where the data are informative about few directions the two rankings can differ. The score is a sensitivity proxy and the two-budget rule is an online heuristic, so neither a bound on the likelihood error nor optimal storage is claimed; the positive-definiteness guarantee of the paper holds for every run under the exact-fp64 assumption.

## References

Apanasovich, T. V., Genton, M. G. and Sun, Y. (2012). A valid Matérn class of cross-covariance functions for multivariate random fields with any number of components. *Journal of the American Statistical Association*, 107, 180-193.

Gneiting, T., Kleiber, W. and Schlather, M. (2010). Matérn cross-covariance functions for multivariate random fields. *Journal of the American Statistical Association*, 105, 1167-1177.

Salvaña, M. L. O. (2026). Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes. In *Proceedings of the SC26 Workshops, HPC for Environmental and Earth Sciences (HPC4EES)*, Chicago, IL, USA, November 15-20, 2026.